# Data Explorer demo

`DataExplorer` opens the data browser for a dataframe: pandas, polars, PyArrow, DuckDB, Spark, and the other libraries [narwhals](https://narwhals-dev.github.io/narwhals/) supports. The tables are in `samples/` (made by `make_samples.py`); this notebook reads them with polars, pandas and DuckDB.

- **Page** with ◀ ▶ (100 rows at a time); **sort** by clicking a column name (again for descending, a third time to stop).
- **Filter** with the funnel next to a column name; active filters are listed in the toolbar (click to edit, × to remove).
- **Select columns** hides and shows columns; drag a column's right edge to resize it (double-click for automatic widths).
- **Reset** removes the filters and shows all columns.
- **Live** / **In memory** (top right): query the data itself, or its first rows read into memory.

In [ ]:
import polars as pl
from jupyter_storage_explorer import DataExplorer, QuerySettings

DataExplorer(pl.scan_delta("samples/lake/customers"))  # a Delta Lake table

Eager dataframes, such as pandas', are always queried in memory (with polars, when it is installed).

In [ ]:
import pandas as pd

orders = pd.read_csv("samples/raw/orders.csv")
DataExplorer(orders, title="orders.csv (pandas)", height="300px")

Larger lazy data starts **Live**: each change of filter, sort or page queries the data again. **In memory** reads its first rows once (`max_memory_rows`, here 50,000), and queries those.

In [ ]:
import duckdb

events = duckdb.sql(
    "SELECT range AS id, 'user' || (range % 997) AS user, range % 7 AS day, random() * 100 AS amount "
    "FROM range(200000)"
)
DataExplorer(events, max_memory_rows=50_000, height="300px")

A function instead of a dataframe is called again by **⟳**, e.g. to read files that changed. A folder of Parquet files (e.g. Spark or Hive output) is one table:

In [ ]:
DataExplorer(lambda: pl.scan_parquet("samples/lake/orders_by_month/**/*.parquet", hive_partitioning=True))

`QuerySettings` gives the filters, sort order, hidden columns and widths to start with. To keep settings made in the widget, print them as text and paste them into your code:

In [ ]:
explorer = DataExplorer(
    pl.scan_csv("samples/raw/orders.csv"),
    settings=QuerySettings(
        filters={"quantity": {"op": "range", "min": 3}},
        sort="unit_price",
        descending=True,
    ),
)
explorer

In [ ]:
print(explorer.settings.to_json())  # after changing the settings in the widget above

In [ ]:
DataExplorer(
    pl.scan_csv("samples/raw/orders.csv"),
    settings=QuerySettings.from_json("""
{
    "filters": {"product": {"op": "contains", "value": "tea", "case_sensitive": false}},
    "sort": {"column": "order_date", "descending": true},
    "hidden": ["customer_id"],
    "widths": {}
}
"""),
)

`explorer.result()` gives the data with the widget's filters, sort order and hidden columns applied, and `on_change` calls a function whenever that changes. Here a chart of the quantity sold per product follows the filters set in the widget: try a range on `order_date` or `unit_price`. The function draws into an `ipywidgets.Output`, shown in the next cell, as what it displays goes to no cell otherwise. `result(all_columns=True)` keeps the columns the chart needs, even if hidden in the widget.

In [ ]:
import ipywidgets
import matplotlib.pyplot as plt

sales = DataExplorer(pl.scan_csv("samples/raw/orders.csv", try_parse_dates=True), height="300px")
chart = ipywidgets.Output()


def draw(explorer):
    by_product = (
        explorer.result(all_columns=True)
        .group_by("product")
        .agg(pl.col("quantity").sum())
        .sort("product")
        .collect()
    )
    with chart:
        chart.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 2.5))
        ax.bar(by_product["product"], by_product["quantity"])
        ax.set_ylabel("Quantity sold")
        plt.show()


sales.on_change(draw)
draw(sales)  # the first chart, before any change
sales

In [ ]:
chart  # redrawn when the filters above change